# Multiple Comparisons for Mixed Models

This notebook mirrors the R *Multiple Comparisons* vignette for `GLMMadaptive`.  The R
version uses the **multcomp** and **emmeans** packages.  Here we use the Python
**marginaleffects** package together with the `glmmadaptive.marginaleffects` adapter.

The adapter wraps a fitted `MixModResults` object so that the full `marginaleffects` API
— `predictions()`, `comparisons()`, `hypotheses()`, `slopes()` — works out of the box.
Predictions are **mean-subject** (population-level fixed effects only).

> **Installation note**: `pip install marginaleffects`

## Setup

In [ ]:
import numpy as np
import pandas as pd

from glmmadaptive import MixedModel
from glmmadaptive.families import Binomial
from glmmadaptive.marginaleffects import wrap_mixmod

from marginaleffects import predictions, comparisons, hypotheses, datagrid

---

## 1  Additive model — pairwise time comparisons

### 1.1  Simulate data

300 subjects × 4 time points, binary outcome with main effects of `sex` and `time`.

In [ ]:
np.random.seed(1234)
n = 300   # subjects
K = 4     # measurements per subject

ids  = np.repeat(np.arange(n), K)
time = pd.Categorical(
    np.tile([f"Time{k}" for k in range(1, K + 1)], n),
    categories=[f"Time{k}" for k in range(1, K + 1)],
)
sex = pd.Categorical(
    np.repeat(np.where(np.arange(n) < n // 2, "male", "female"), K),
    categories=["male", "female"],
)

df = pd.DataFrame({"id": ids, "time": time, "sex": sex})

X = pd.get_dummies(df[["sex", "time"]], drop_first=True).astype(float)
X.insert(0, "Intercept", 1.0)
betas_true = np.array([-2.13, 1.0, 1.2, -1.2, 1.2])
b = np.random.normal(0, 1, n)[ids]
eta = X.values @ betas_true + b
df["y"] = np.random.binomial(1, 1.0 / (1.0 + np.exp(-eta)))

df.head(8)

### 1.2  Fit additive model

In [ ]:
fm = MixedModel(
    "y ~ sex + time",
    random="~ 1 | id",
    data=df,
    family=Binomial(),
).fit()

print(fm.summary())

### 1.3  Wrap for marginaleffects

In [ ]:
mfm = wrap_mixmod(fm)

### 1.4  Pairwise time contrasts on the log-odds scale

R uses `multcomp::glht(fm, linfct = mcp(time = "Tukey"))`.  Python uses `hypotheses()`
with explicit contrast strings.  Parameters (0-indexed): `b0`=Intercept, `b1`=sex,
`b2`=Time2, `b3`=Time3, `b4`=Time4.

In [ ]:
pairwise_time = hypotheses(mfm, hypothesis=[
    "b2 = 0",        # Time2 vs Time1
    "b3 = 0",        # Time3 vs Time1
    "b4 = 0",        # Time4 vs Time1
    "b2 - b3 = 0",   # Time2 vs Time3
    "b2 - b4 = 0",   # Time2 vs Time4
    "b3 - b4 = 0",   # Time3 vs Time4
])

pairwise_time

Estimates are log-odds differences; p-values use the Wald normal approximation.

### 1.5  Predictions on a time grid

In [ ]:
time_grid = datagrid(
    model=mfm,
    time=["Time1", "Time2", "Time3", "Time4"],
)
p_time = predictions(mfm, newdata=time_grid)
p_time.select(["time", "estimate", "conf_low", "conf_high"])

---

## 2  Interaction model — estimated marginal means

### 2.1  Fit model with sex × time interaction

In [ ]:
gm = MixedModel(
    "y ~ sex * time",
    random="~ 1 | id",
    data=df,
    family=Binomial(),
).fit()

print(gm.summary())

### 2.2  Estimated marginal means for sex × time

R uses `emmeans(gm, ~ sex | time)`.  Python builds a prediction grid explicitly.

In [ ]:
mgm = wrap_mixmod(gm)

sex_time_grid = datagrid(
    model=mgm,
    sex=["male", "female"],
    time=["Time1", "Time2", "Time3", "Time4"],
)

emm = predictions(mgm, newdata=sex_time_grid)
emm.select(["sex", "time", "estimate", "conf_low", "conf_high"])

### 2.3  Pairwise sex comparisons within each time point

R uses `pairs(gm_mc)`.  Python uses `comparisons()` with `by="time"` on the grid.

In [ ]:
cmp_sex = comparisons(
    mgm,
    variables={"sex": "pairwise"},
    by="time",
    newdata=sex_time_grid,
)
cmp_sex.select(["contrast", "time", "estimate", "std_error", "p_value",
                "conf_low", "conf_high"])

Each row gives the female − male difference in predicted probability at one time point,
with a delta-method standard error.

---

## 3  Notes on differences from the R vignette

| Feature | R | Python |
|---------|---|--------|
| Pairwise log-odds tests | `multcomp::glht` + Tukey MVN correction | `hypotheses()` + Wald normal approx. |
| Estimated marginal means | `emmeans(gm, ~ sex \| time)` | `datagrid()` + `predictions()` |
| Pairwise response-scale comparisons | `pairs()` | `comparisons(by=...)` on a grid |
| Sandwich SEs | `vcov. = vcov(fm, sandwich=TRUE)` | `wrap_mixmod(fit, vcov="sandwich")` |
| Prediction type | Subject-specific or marginal | Mean-subject only |

### Key limitation: subject-specific predictions

`marginaleffects` propagates uncertainty via the delta method on the fixed-effect
coefficient vector.  Subject-specific predictions (which add empirical Bayes random
effects) cannot be supported because they require group membership information.
Use `fit.predict(type_="subject_specific")` directly for those.